#Initialization

In [0]:
%python
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import trim, col
from pyspark.sql.window import Window

# Read bronze table crm_sales_details

In [0]:
%python
df = spark.table("workspace.bronze.crm_sales_details")

#Silver transformation

##Trimming

In [0]:
%python
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))


## Checking datatypes 

In [0]:
%python
df.printSchema()

##Cleaning Dates

In [0]:
%python
df = (
    df
    .withColumn(
        "sls_order_dt",
        F.when(
            (F.col("sls_order_dt") == 0) | (F.length(F.col("sls_order_dt")) != 8),
            None
        ).otherwise(
            F.to_date(F.col("sls_order_dt").cast("string"), "yyyyMMdd")
        )
    )
    .withColumn(
        "sls_ship_dt",
        F.when(
            (F.col("sls_ship_dt") == 0) | (F.length(F.col("sls_ship_dt")) != 8),
            None
        ).otherwise(
            F.to_date(F.col("sls_ship_dt").cast("string"), "yyyyMMdd")
        )
    )
    .withColumn(
        "sls_due_dt",
        F.when(
            (F.col("sls_due_dt") == 0) | (F.length(F.col("sls_due_dt")) != 8),
            None
        ).otherwise(
            F.to_date(F.col("sls_due_dt").cast("string"), "yyyyMMdd")
        )
    )
)

##Sales and Price corrections

In [0]:
%python
df = (
    df
    .withColumn(
        "sls_price",
        F.when(
            col("sls_price").isNull() | (col("sls_price") <= 0),
            F.when(
                col("sls_quantity") !=0,
                col("sls_sales") / col("sls_quantity")
            ).otherwise(None)
            ).otherwise(col("sls_price")))
    )

##Renaming columns

In [0]:
%python
RENAME_MAP = {
   "sls_ord_num": "order_number",
   "sls_prd_key": "product_number",
   "sls_cust_id": "customer_id",
   "sls_order_dt": "order_date",
   "sls_ship_dt": "ship_date",
   "sls_due_dt": "due_date",
   "sls_sales": "sales_amount",
   "sls_quantity": "quantity",
   "sls_price": "price"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

##Sanity checks of dataframe

In [0]:
%python
df.limit(10).display()

#Writing silver table

In [0]:
%python
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_sales")

##Sanity checks of silver table

In [0]:
SELECT * FROM silver.crm_sales LIMIT 10